# Week 1 AI/ML: Temporal Split, Feature Engineering & Baseline Models
**Smart Renewable Energy Forecasting & Battery Arbitrage System**

This notebook covers the **Week 1 AI/ML Track** deliverables:
1. **Data Access:** Programmatic loading of post-EDA hourly handoff datasets ().
2. **Time-Series Feature Engineering:** Solar zenith angle & elevation geometry, rolling irradiance averages, target lags, cyclical calendar features.
3. **Nighttime Zero-Inflation:** Strict physical enforcement of zsh.0	ext{ kW}$ output during night hours to avoid negative/phantom predictions.
4. **Chronological Time-Based Split:** Strict temporal train/test split (last 7 days = 168 hours test horizon, no data leakage).
5. **Baseline Benchmarks:** Persistence (24h lag), Diurnal Climatological Mean, and Linear/Ridge Regression.
6. **Evaluation Metrics:** Daylight RMSE, Daylight MAPE, nRMSE, and ^2$.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

# Ensure project root is on sys.path
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data.loader import load_processed_handoff
from src.features.engineering import create_feature_pipeline, apply_nighttime_zero_inflation
from src.models.baseline import run_week1_baseline_benchmark, temporal_train_test_split
from src.models.metrics import evaluate_forecast

pd.set_option("display.max_columns", None)
print("Project root resolved to:", project_root)

## 1. Load Processed Post-EDA Datasets
We load the post-EDA hourly handoff dataset for Solar Plant 1 (68,778 fifteen-minute inverter records across 34 days).

In [ ]:
df_plant1 = load_processed_handoff(plant_id=1, freq="15min")
print("Dataset Shape:", df_plant1.shape)
display(df_plant1.head())

## 2. Feature Engineering Pipeline
We construct physical and temporal features:
- **Solar Zenith Angle ($\cos(\theta_z)$) & Elevation Angle:** Calculated using the exact geographic coordinates (14.80°N, 78.18°E) and Spencer solar geometry equations.
- **Rolling Irradiance:** 3-hour and 6-hour backward-looking rolling means and standard deviations of GHI and sensor irradiation.
- **Autoregressive Lags:** -1	ext{h}$, -24	ext{h}$, -48	ext{h}$.
- **Cyclical Calendar Features:** Sin/Cos transformed hours and days of year.

In [ ]:
df_features = create_feature_pipeline(df_plant1, plant_id=1, target_col="ac_power", include_lags=True)
print("Feature Matrix Shape:", df_features.shape)
print("Engineered Columns:", [c for c in df_features.columns if c not in df_plant1.columns])
display(df_features[["time", "ac_power", "solar_elevation_deg", "cos_zenith", "ghi_rolling_mean_3h", "ac_power_lag_24h", "is_daylight"]].head(10))

## 3. Strict Chronological Train/Test Split
In time-series modeling, random splitting leaks future information into past predictions. We perform a strict temporal split where the final 7 days (168 hours) serve as the test set.

In [ ]:
train_df, test_df = temporal_train_test_split(df_features, test_days=7)
print(f"Train set: {len(train_df)} hours (from {train_df['time'].min()} to {train_df['time'].max()})")
print(f"Test set:  {len(test_df)} hours (from {test_df['time'].min()} to {test_df['time'].max()})")

## 4. Benchmark Baseline Models
We benchmark 4 models:
1. **Persistence (24h Lag):** Assumes tomorrow’s solar output at hour $ matches yesterday’s.
2. **Diurnal Hourly Mean:** Historical average output for each hour of the day.
3. **Linear Regression:** Standardized linear model using meteorological features.
4. **Ridge Regression:** Regularized linear model with $ penalty.

In [ ]:
metrics_df, test_results, predictions = run_week1_baseline_benchmark(df_plant1, plant_id=1, test_days=7)
print("=== Plant 1 Baseline Performance (Sorted by Daylight RMSE) ===")
display(metrics_df)

## 5. Visualizing Actual vs. Baseline Predictions (7-Day Test Window)
We plot actual AC generation against Ridge Regression and Persistence predictions across the 7-day out-of-sample evaluation period.

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=test_results["time"], y=test_results["ac_power"], mode="lines", name="Actual Generation (kW)", line=dict(color="black", width=2.5)))
fig.add_trace(go.Scatter(x=test_results["time"], y=predictions["Ridge Regression"], mode="lines", name="Ridge Baseline", line=dict(color="#1E90FF", width=2, dash="dash")))
fig.add_trace(go.Scatter(x=test_results["time"], y=predictions["Persistence (24h Lag)"], mode="lines", name="Persistence (24h)", line=dict(color="#FFA500", width=1.5, dash="dot")))
fig.update_layout(title="Plant 1 Out-of-Sample Test Window: Actual vs Baseline Predictions", xaxis_title="Timestamp", yaxis_title="AC Power (kW)", hovermode="x unified", template="plotly_dark", height=500)
fig.show()